# Help Desk Ticket SLA and Volume Forecasting

This capstone uses `data/raw/issues.csv` for two integrated workstreams: **(1) retrospective comparison of elapsed resolution time with project-configured SLA references by recorded priority; and (2) separate 7- and 30-day daily Ticket-volume forecasting experiments for capacity-planning research.** The 30-day forecast is a daily path, not one monthly total. The primary analysis includes `issue_type == Ticket` and creation dates on/after 2016-01-01 UTC, matching the period described in the accompanying source documentation.

**Important source caveat:** the CSV contains creation dates before 2016 while the documentation's stated range says it started in 2016. Those records are profiled but excluded from the primary scope. Dates without scoped Ticket rows are treated as zero arrivals in the forecasting series under an explicit completeness assumption that has not been independently verified. The source ends in March 2023 and cannot establish current operational performance.

The configured SLA references are project assumptions: Blocker/Highest → Critical (4h), High (8h), Medium (24h), Low/Lowest → Low (24h); `unknown` is excluded. The measured duration is calendar wall-clock time between creation and recorded resolution—not a verified business-hours SLA clock. Results are retrospective, not contractual compliance or a live warning model.

**Shared retrospective result across this notebook sequence:** among 16,735 eligible completed Tickets, 3,333 (19.9%) met the configured reference. By mapped priority: Critical 524/2,124 (24.7%); High 576/3,146 (18.3%); Medium 2,161/11,161 (19.4%); Low 72/304 (23.7%). Eligibility requires exact Ticket type, documented 2016+ scope, `Done`, known mapped priority, and valid nonnegative elapsed duration. Notebook 02 shows the detailed comparison; these same counts and caveats are the project-wide retrospective baseline.


## 07 | Deployment readiness, monitoring and experiment tracking

Notebook 03 selects a model by validation MAE and refits it on full history as `models/final_volume_forecast_model.joblib` (Section: model persistence). This notebook is where that single artifact is (1) registered as a tracked MLflow run so its parameters, metrics and files have auditable lineage rather than only static JSON, and (2) checked against the same historical error-monitoring and release-gate questions any capacity-planning forecast needs before a pilot. Keep this predictive workstream distinct from the retrospective SLA baseline (3,333/16,735 eligible Tickets met the configured reference; 19.9%). This capstone remains an offline backtest; there is no scheduled feed, current source data, production endpoint or staffing integration.


### Experiment tracking (MLflow)

Log the notebook 03 selection outcome and the persisted model artifact as one MLflow run: parameters (model name, selection rule, primary horizon, training window), metrics (validation MAE and the primary-horizon test MAE/RMSE/WAPE/sMAPE/coverage), and the artifact files themselves (`.joblib` + metadata + selection JSON). Tracking metadata is a local SQLite database (`mlflow.db`, gitignored), with run artifacts under `mlruns/` (also gitignored) — this MLflow version requires a database backend rather than a plain folder. Re-run this cell any time after notebook 03 to record the current artifact's lineage; inspect it with `mlflow ui --backend-store-uri sqlite:///mlflow.db` from the project root.


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.issues_capstone import log_experiment_to_mlflow
run_summary = log_experiment_to_mlflow()
print(f"Experiment: {run_summary['experiment_name']}  |  Run: {run_summary['run_id']}")
print(f"Tracking URI: {run_summary['tracking_uri']}")
print('Params:', run_summary['logged_params'])
print('Metrics:', run_summary['logged_metrics'])


Experiment: helpdesk_volume_forecast  |  Run: 269964acc5434805acfaeec1245747bd
Tracking URI: sqlite:///C:\Users\acbitor\PycharmProjects\PythonProject for AIM\mlflow.db
Params: {'model_name': 'ETS (weekly additive)', 'selection_rule': 'minimum validation MAE only; test not used for selection', 'primary_horizon_days': 30, 'trained_on_rows': 2628, 'trained_on_date_start': '2016-01-03', 'trained_on_date_end': '2023-03-14', 'validation_calendar_days': 365, 'test_calendar_days': 365}
Metrics: {'validation_mae': 4.393814913939992, 'test_mae': 4.672304845138525, 'test_rmse': 7.197035877605334, 'test_wape': 0.275066188757133, 'test_smape': 0.4132757395207179, 'test_coverage_90_interval': 0.8770833333333333}


## Historical monitoring

Independent of experiment tracking, roll the primary-horizon selected model's validation/test predictions up by month to see whether error and bias are stable across the backtest, or drift toward the end of the source period. This is a diagnostic on past data, not a live monitoring feed.


In [2]:
from IPython.display import display
from src.issues_capstone import run_monitoring_plan
result = run_monitoring_plan()
print(result['plan'])
display(result['historical_monitoring'])


{'status': 'offline historical monitoring only; no live feed or scheduled job', 'monitor': ['Daily source row counts and date completeness', 'Weekly and monthly actual-versus-forecast signed bias and MAE', 'Validation-calibrated interval coverage on future data', 'Priority-reference mapping and resolution-date quality', 'Data/schema drift and sustained ticket mix changes'], 'response': 'Pause operational use on source gaps, sustained under-forecasting, interval undercoverage, or failed owner-approved tolerance; investigate before retraining.', 'human_review': 'Capacity planner approves actions and records overrides; forecast does not allocate staffing automatically.'}


,split,forecast_month,days,actual_total,forecast_total,mean_signed_error,mae
0,test,2022-03,15,265.0,237.328930,1.844738,4.882915
1,test,2022-04,111,1504.0,1499.840639,0.037472,3.474601
2,test,2022-05,134,1753.0,1875.267575,-0.912445,5.565101
3,test,2022-06,128,2128.0,1764.940519,2.836402,4.282750
4,test,2022-07,132,1731.0,1992.760304,-1.983033,6.073067
5,test,2022-08,134,2185.0,2065.100087,0.894775,3.316127
6,test,2022-09,128,2271.0,1944.559029,2.550320,4.967771
7,test,2022-10,133,2324.0,2256.664032,0.506285,3.759649
8,test,2022-11,129,2432.0,2307.329215,0.966440,4.462439
9,test,2022-12,132,2469.0,2426.191127,0.324310,5.438127


## Release gates

Before a pilot: verify source completeness and the zero-day interpretation; confirm the priority mapping, SLA clock and applicability with owners; establish current data access; agree tolerances in advance; evaluate on a genuinely future period; measure interval coverage and subgroup errors where appropriate; and require a planner to review and log decisions. A tracked MLflow run and a stable historical-monitoring trend are necessary evidence for this gate, not sufficient approval on their own. Stop or investigate on source gaps, sustained under-forecasting, interval undercoverage or excessive drift. No deployment approval is established by these historical results.


## In summary

This notebook takes the saved forecasting model from notebook 03 and treats it the way a real deployment would: it logs the model's settings and performance to MLflow, a tool for tracking machine learning experiments, so there is a clear record of what was built and how well it performed, checks whether its accuracy has stayed stable over time, and lists the concrete steps, data checks, owner sign offs, a trial run, that would need to happen before this could be trusted in daily operations. Nothing here claims the system is live or production ready.